In [7]:
# NNSE Function-based Implementation for Tyson Model
# Implements a vector-based mutation and permutation algorithm

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import copy

# ============================================================================
# === CONFIGURATION VARIABLES ===
# ============================================================================

# Simulation settings
N_STEPS = 100  # Number of mutation steps to run
SIGMA = 0.05  # Standard deviation for Gaussian mutations in normalized space
N_Vec = 51  # Number of bins for binning squared differences
MAX_VALUE = 70  # Maximum log value for bin thresholds (logspace goes from 0 to this)
K_INITIAL = int(N_Vec // 50)  # Number of top positions to fill initially (top half)
T_START = 0.0  # Simulation start time
T_END = 500.0  # Simulation end time
N_TIME_POINTS = 501  # Number of time points

# Define bin thresholds using logspace (equally spaced in log space)
# y0, y1, ..., yN_BINS: thresholds for binning
bin_thresholds = np.linspace(0, MAX_VALUE, N_Vec + 1) + MAX_VALUE / N_Vec
  # y0, y1, ..., yN_BINS, basically sets y0 to the value of y1 because function will likely never be zero

# ============================================================================
# === BASE PARAMETERS (p0) ===
# ============================================================================

p0 = {
    "k1_aa_over_CT": 0.015,
    "k2": 0.0,
    "k3_CT": 200.0,
    "k4": 180.0,
    "k4prime": 0.018,
    "k5_minusP": 0.0,
    "k6": 1.0,
    "k7": 0.6,
    "k8_minusP": 100.0,
    "k9": 50.0,
    "CT": 1.0
}

# Parameters to vary
param_names = [
    "k1_aa_over_CT",
    "k3_CT",
    "k4",
    "k4prime",
    "k6",
    "k7"
]

p0_vec = np.array([p0[name] for name in param_names])
n_params = len(param_names)

# Time evaluation array
t_eval = np.linspace(T_START, T_END, N_TIME_POINTS)

# ============================================================================
# === TYSON MODEL DEFINITION ===
# ============================================================================

CT = p0["CT"]

def F_M(M, p):
    """Helper function for M-dependent rate"""
    return p["k4prime"] + p["k4"] * (M / p["CT"])**2

def f_rhs(t, x, p):
    """Right-hand side of the ODE system"""
    # x = [C2, CP, pM, M, Y, YP]
    C2, CP, pM, M, Y, YP = x
    k3 = p["k3_CT"] / p["CT"]
    k1 = p["k1_aa_over_CT"] * p['CT']
    dC2 = p["k6"] * M - p["k8_minusP"] * C2 + p["k9"] * CP
    dCP = -k3 * CP * Y + p["k8_minusP"] * C2 - p["k9"] * CP
    dpM = k3 * CP * Y - pM * F_M(M, p) + p["k5_minusP"] * M
    dM  = pM * F_M(M, p) - p["k5_minusP"] * M - p["k6"] * M
    dY  = k1 - p["k2"] * Y - k3 * CP * Y
    dYP = p["k6"] * M - p["k7"] * YP
    return np.array([dC2, dCP, dpM, dM, dY, dYP])

def simulate_at_params(p_local, t_eval):
    """Simulate the ODE system with given parameters"""
    y0 = np.array([0.9, 0.05, 0.0, 0.005, 0.3, 0.0])
    sol = solve_ivp(lambda tt, xx: f_rhs(tt, xx, p_local), 
                    (t_eval[0], t_eval[-1]), y0,
                    method='BDF', t_eval=t_eval, rtol=1e-6, atol=1e-8)
    if not sol.success:
        raise RuntimeError("Integrator failed: " + sol.message)
    return sol.t, sol.y

def compute_obs(X):
    """Compute observables: YT/CT and M/CT"""
    if X.ndim == 1:
        C2, CP, pM, M, Y, YP = X
        YT = Y + YP + pM + M
        return YT / CT, M / CT
    else:
        C2, CP, pM, M, Y, YP = X
        YT = Y + YP + pM + M
        return YT / CT, M / CT

# ============================================================================
# === REFERENCE SIMULATION (p0) ===
# ============================================================================

print("Running reference simulation with p0...")
t0, y0 = simulate_at_params(p0, t_eval)
YT0, M0 = compute_obs(y0)
print(f"✓ Reference simulation complete")

# ============================================================================
# === SIM FUNCTION ===
# ============================================================================

def sim(P_vec):
    """
    Simulate with parameter vector P and compute squared difference with p0.
    Returns the squared difference (f(xi) value).
    """
    # Convert parameter vector to dictionary
    p_local = copy.deepcopy(p0)
    for i, name in enumerate(param_names):
        p_local[name] = P_vec[i]
    
    # Run simulation
    try:
        t, y = simulate_at_params(p_local, t_eval)
        YT, M = compute_obs(y)
        
        # Interpolate reference to match time points
        YT0_interp = np.interp(t, t0, YT0)
        M0_interp = np.interp(t, t0, M0)
        
        # Compute squared differences
        diff_YT_sq = (YT - YT0_interp)**2
        diff_M_sq = (M - M0_interp)**2
        
        # Integrate squared differences
        integral_YT_sq = np.trapz(diff_YT_sq, t)
        integral_M_sq = np.trapz(diff_M_sq, t)
        squared_diff = integral_YT_sq + integral_M_sq
        
        return squared_diff
        
    except Exception as e:
        # If simulation fails, return infinity
        print(f"Warning: Simulation failed: {e}")
        return np.inf

print(f"✓ Configuration complete")
print(f"  Parameters: {param_names}")
print(f"  Number of parameters: {n_params}")
print(f"  Bin thresholds (y0, ..., y{N_Vec}): [{bin_thresholds[0]:.4e}, ..., {bin_thresholds[-1]:.4e}]")


Running reference simulation with p0...
✓ Reference simulation complete
✓ Configuration complete
  Parameters: ['k1_aa_over_CT', 'k3_CT', 'k4', 'k4prime', 'k6', 'k7']
  Number of parameters: 6
  Bin thresholds (y0, ..., y51): [1.3725e+00, ..., 7.1373e+01]


In [8]:
# ============================================================================
# === TYSONFUNC: MUTATION AND PERMUTATION FUNCTION ===
# ============================================================================

def TysonFunc(X_list, fX_list):
    """
    Mutate each parameter vector, evaluate, reject if worse, then permute.
    
    Args:
        X_list: List of parameter vectors [x0, x1, ..., xn] where each xi is a numpy array
        fX_list: List of function values [f(x0), f(x1), ..., f(xn)]
    
    Returns:
        v_list: List of parameter vectors after mutation and permutation [v0, v1, ..., vn]
        fv_list: List of function values [f(v0), f(v1), ..., f(vn)]
        fX_prime_list: List of function values after mutation but before permutation [f(x'0), f(x'1), ..., f(x'n)]
        swaps: List of (i, j) tuples indicating which positions were swapped
    """
    n = len(X_list)
    
    # Step 1: Mutate each xi
    X_prime_list = []
    fX_prime_list = []
    
    for i in range(n):
        xi = X_list[i]
        fxi = fX_list[i]
        
        # Skip empty positions
        if xi is None or fxi is None:
            X_prime_list.append(None)
            fX_prime_list.append(None)
            continue
        
        # Normalize parameters: u_i = p_i / (2 * p0_i) so each lies in [0,1]
        u_vec = xi / (2.0 * p0_vec)
        
        # Apply Gaussian mutation in u-space
        u_mutated = u_vec + np.random.normal(0, SIGMA, size=n_params)
        
        # Wrap around boundaries [0, 1] with periodic boundary conditions
        u_mutated = u_mutated % 1.0
        
        # Map back to parameter space: p_i = 2 * p0_i * u_i
        xi_prime = 2.0 * p0_vec * u_mutated
        
        # Evaluate mutated parameter: f(x'i) := sim(x'i)
        fxi_prime = sim(xi_prime)
        
        # Step 2: Reject x'i if f(x'i) > y_i, otherwise accept
        if fxi_prime > bin_thresholds[i]:
            # Reject: keep original
            X_prime_list.append(xi.copy())
            fX_prime_list.append(fxi)
        else:
            # Accept: use mutated
            X_prime_list.append(xi_prime)
            fX_prime_list.append(fxi_prime)
    
    # Step 3: Permutation step
    # For each position i from n-1 down to 1 (0-indexed: n-1, n-2, ..., 1)
    # If f(x'_i) <= y_{i-1}, swap position i with position i-1
    v_list = X_prime_list.copy()
    fv_list = fX_prime_list.copy()
    
    # Track swaps
    swaps = []  # List of (i, j) tuples for swaps
    
    for i in range(n-1, 0, -1):  # i from n-1 down to 1
        # Skip if either position is empty
        if fv_list[i] is None or fv_list[i-1] is None:
            continue
        
        if fv_list[i] <= bin_thresholds[i-1]:  # f(x'_i) <= y_{i-1}
            # Swap position i with position i-1
            if v_list[i] is not None and v_list[i-1] is not None:
                v_list[i], v_list[i-1] = v_list[i-1].copy(), v_list[i].copy()
            else:
                v_list[i], v_list[i-1] = v_list[i-1], v_list[i]
            fv_list[i], fv_list[i-1] = fv_list[i-1], fv_list[i]
            swaps.append((i, i-1))  # Record the swap
    
    return v_list, fv_list, fX_prime_list, swaps

print("✓ TysonFunc defined")


✓ TysonFunc defined
